# Manual tagging validation export

Exports a random, stratified sample of already-processed articles to Excel
for manual review of the **sentiment**, **category**, and **NER** stages
(summarization is out of scope for this notebook).

**Sampling**: up to `TARGET_PER_CATEGORY` (200) articles per category label
(the 9-slug taxonomy + `other`), each category's sample balanced as evenly
as possible across the three sentiment classes (positive/negative/neutral).
A `(category, sentiment)` cell with fewer real articles than its target is
taken in full and disclosed in the shortfall report below, never topped up
from a different cell -- silently doing so would break both the "200 per
category" and "balanced sentiment" contracts at once.

**Setup**:
```bash
uv sync --group notebook   # pandas + openpyxl + ipykernel
```
Needs `$DATABASE_URL`/`$SOURCE_DATABASE_URL` configured the same way the
pipeline does (`.env`, see `docs/db-topology.md`) -- this notebook only
*reads* both stores, it never writes.

In [ ]:
import sys
from pathlib import Path

# VSCode/Jupyter may run this notebook with its working directory set to
# scripts/ or to the repo root, depending on settings -- `__file__` isn't
# defined in a notebook, unlike every other entrypoint's sys.path bootstrap
# (constitution: Project structure #2), so handle both rather than assume one.
_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "scripts" else _cwd
assert (REPO_ROOT / "src" / "news_nlp").is_dir(), (
    f"Couldn't locate src/news_nlp from {REPO_ROOT} -- run this notebook "
    "with its working directory set to the repo root or to scripts/."
)
sys.path.insert(0, str(REPO_ROOT / "src"))

In [ ]:
import json
import random
from collections import defaultdict

import pandas as pd
from dotenv import load_dotenv
from portfolio_common.db import in_clause

import news_nlp as db

# pipeline.py normally owns the one load_dotenv() call (constitution: Project
# structure #6) -- this notebook deliberately doesn't import pipeline.py (it
# would pull in torch/transformers for zero benefit here, since this notebook
# runs no model), so it loads .env itself instead.
load_dotenv(REPO_ROOT / ".env")

In [ ]:
TARGET_PER_CATEGORY = 200
SENTIMENT_CLASSES = ["positive", "negative", "neutral"]
CATEGORY_LABELS = [*db.CATEGORY_SLUGS, db.OTHER_LABEL]
ENTITY_TYPES = ["ORG", "PER", "LOC"]
# Fixed seed for a reproducible sample -- TASKS.md T-037 found an unseeded
# resample can badly skew a data-quality review.
SEED = 42
OUTPUT_PATH = REPO_ROOT / "data" / "tagging_validation.xlsx"

In [ ]:
conn = db.connect_pipeline()
db.require_source_text(conn)  # fail fast if SOURCE has no usable body_text
rel = conn.articles_rel
print(f"articles_rel = {rel!r}")

## Phase 1 -- lightweight population query

Only `article_id`/`sentiment`/`category` for every article that has been
fully processed by both the sentiment and category stages -- `body_text`
isn't selected here, so this stays cheap even against a large corpus. NER
isn't a stratification key (see the scope note above), so
`article_entities` isn't touched here either -- entities are fetched later,
only for the ~2,000 sampled articles.

In [ ]:
population_sql = f"""
    SELECT a.id AS article_id, s.label AS sentiment_label, c.label AS category_label
    FROM {rel}.articles a
    JOIN article_sentiment s ON s.article_id = a.id
    JOIN article_category c ON c.article_id = a.id
    WHERE a.body_text IS NOT NULL AND TRIM(a.body_text) != ''
"""  # noqa: S608 -- `rel` is Allowlist-checked via conn.articles_rel, never caller input
cur = conn.execute(population_sql)
population = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
print(f"{len(population)} fully-processed articles available (sentiment + category present)")
population[["category_label", "sentiment_label"]].value_counts().sort_index()

In [ ]:
def stratified_sample(population: pd.DataFrame, seed: int) -> tuple[pd.DataFrame, list[dict]]:
    """200 per category label, balanced as evenly as possible across the
    three sentiment classes within each category. A cell short on real data
    is taken in full and disclosed in the shortfall report -- never topped
    up from a different sentiment class or category, which would silently
    break the "balanced"/"200 per category" contract instead of reporting
    a real data gap (matching this project's own disclosed-gaps convention,
    e.g. experiments/README.md's "disclosed gaps" section)."""
    rng = random.Random(seed)  # noqa: S311 -- sample selection, not cryptography
    picked_frames = []
    report = []
    for category in CATEGORY_LABELS:
        cat_pop = population[population["category_label"] == category]
        base, remainder = divmod(TARGET_PER_CATEGORY, len(SENTIMENT_CLASSES))
        for i, sentiment in enumerate(SENTIMENT_CLASSES):
            target = base + (1 if i < remainder else 0)
            cell = cat_pop[cat_pop["sentiment_label"] == sentiment]
            n = min(target, len(cell))
            if n > 0:
                sampled_ids = rng.sample(list(cell["article_id"]), n)
                picked_frames.append(cell[cell["article_id"].isin(sampled_ids)])
            report.append(
                {
                    "category": category,
                    "sentiment": sentiment,
                    "requested": target,
                    "available": len(cell),
                    "sampled": n,
                }
            )
    sample = pd.concat(picked_frames, ignore_index=True) if picked_frames else population.iloc[0:0]
    return sample, report


sample_ids_df, shortfall_report = stratified_sample(population, SEED)
report_df = pd.DataFrame(shortfall_report)
short = report_df[report_df["sampled"] < report_df["requested"]]

print(f"Sampled {len(sample_ids_df)} articles across {len(CATEGORY_LABELS)} categories.")
if not short.empty:
    print(f"{len(short)} (category, sentiment) cell(s) came up short of their target:")
    display(short)
else:
    print("Every (category, sentiment) cell met its target.")

In [ ]:
sampled_article_ids = sample_ids_df["article_id"].tolist()
placeholders = in_clause(sampled_article_ids)

detail_sql = f"""
    SELECT a.id AS article_id, a.body_text,
           s.label AS sentiment, s.score AS sentiment_score,
           c.label AS category, c.score AS category_score
    FROM {rel}.articles a
    JOIN article_sentiment s ON s.article_id = a.id
    JOIN article_category c ON c.article_id = a.id
    WHERE a.id IN {placeholders}
"""  # noqa: S608 -- `rel` is Allowlist-checked; article ids are bound params below, never interpolated
cur = conn.execute(detail_sql, sampled_article_ids)
details = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
print(f"{len(details)} rows fetched")

In [ ]:
entities_sql = f"""
    SELECT article_id, entity_type, text
    FROM article_entities
    WHERE article_id IN {placeholders}
"""  # noqa: S608
cur = conn.execute(entities_sql, sampled_article_ids)
entities_rows = cur.fetchall()

unexpected_types = {row[1] for row in entities_rows} - set(ENTITY_TYPES)
if unexpected_types:
    print(f"WARNING: unexpected entity_type value(s), not in {ENTITY_TYPES}: {unexpected_types}")

entities_by_article: dict[int, dict[str, list[str]]] = defaultdict(
    lambda: {t: [] for t in ENTITY_TYPES}
)
for article_id, entity_type, text in entities_rows:
    bucket = entities_by_article[article_id].setdefault(entity_type, [])
    if text not in bucket:  # de-duplicate repeated mentions, keep first-seen order
        bucket.append(text)

details["entities"] = details["article_id"].map(
    lambda aid: json.dumps(
        {t: entities_by_article.get(aid, {}).get(t, []) for t in ENTITY_TYPES}, ensure_ascii=False
    )
)
print(
    f"{len(entities_rows)} entity mentions across "
    f"{len(entities_by_article)} of the {len(details)} sampled articles"
)

In [ ]:
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

final = details[
    [
        "article_id",
        "body_text",
        "sentiment",
        "sentiment_score",
        "category",
        "category_score",
        "entities",
    ]
].copy()
# Shuffle so rows aren't grouped by category/sentiment in the sheet -- a
# reviewer working straight down the sheet shouldn't be able to guess the
# next row's label from its position.
final = final.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Real crawled body_text can carry control characters (PDF/HTML extraction
# artifacts) that Excel's XML format rejects outright -- openpyxl raises
# IllegalCharacterError deep into the write, after most rows already
# succeeded, rather than failing at the start. Strip them from every text
# column up front instead of discovering this on a ~2,000-row run.
for col in ("body_text", "entities"):
    final[col] = final[col].map(
        lambda v: ILLEGAL_CHARACTERS_RE.sub("", v) if isinstance(v, str) else v
    )

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
final.to_excel(OUTPUT_PATH, index=False, sheet_name="tagging_validation")
print(f"Wrote {len(final)} rows to {OUTPUT_PATH}")
final.head(3)

In [ ]:
db.detach_source(conn)
conn.close()

## Output

`data/tagging_validation.xlsx` (git-ignored, like the rest of `data/`), one
sheet (`tagging_validation`), columns: `article_id` (added for traceability
back to the DB row -- not one of the originally requested columns, but a
validation export with no way to look a row back up isn't reviewable),
`body_text`, `sentiment`, `sentiment_score`, `category`, `category_score`,
`entities` (a JSON string per row, `{"ORG": [...], "PER": [...], "LOC": [...]}`,
de-duplicated per entity type).

If any `(category, sentiment)` cell came up short above, the sheet has
fewer than `10 * TARGET_PER_CATEGORY` rows -- expected when the real corpus
doesn't have 200 processed articles at some label/sentiment combination yet,
not a bug in the sampling.